In [ ]:
import tensorflow as tf

print("TensorFlow :", tf.__version__)
print("GPU détecté :", tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/DermaVision', exist_ok=True)
print("Drive monté !")

In [ ]:
# Met à jour kaggle vers la version qui supporte les clés KGA_
!pip install --upgrade kaggle -q

import json, os

# Tes valeurs depuis le fichier kaggle.json téléchargé
KAGGLE_USERNAME = "XXXXXXXXX"   # ← remplacer
KAGGLE_KEY      = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"    # ← coller clé complète

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)

os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!cat ~/.kaggle/kaggle.json
print("\nKaggle configuré !")

In [ ]:
!kaggle datasets download -d maysee/mushrooms-classification-common-genuss-images --force
!unzip -q mushrooms-classification-common-genuss-images.zip -d mushrooms
print("Dataset prêt !")
!ls mushrooms/Mushrooms

In [ ]:
import os

DATA_DIR = 'mushrooms/Mushrooms'
classes = sorted(os.listdir(DATA_DIR))
print(f"{len(classes)} genres trouvés :", classes)
for cls in classes:
    n = len(os.listdir(os.path.join(DATA_DIR, cls)))
    print(f"  {cls:<15} : {n} images")

In [ ]:
import pandas as pd

rows = []
for cls in classes:
    folder = os.path.join(DATA_DIR, cls)
    for img in os.listdir(folder):
        if img.lower().endswith(('.jpg', '.jpeg', '.png')):
            rows.append({'path': os.path.join(folder, img), 'label': cls})

df = pd.DataFrame(rows)
print(f"Total images : {len(df)}")
print(df['label'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
print(f"Train : {len(train_df)} images")
print(f"Val   : {len(val_df)} images")

In [ ]:
import tensorflow as tf

IMG_SIZE   = 224
BATCH_SIZE = 32

train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)
val_gen = tf.keras.preprocessing.image.ImageDataGenerator()

train_flow = train_gen.flow_from_dataframe(
    train_df, x_col='path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'   # ← multi-classes
)
val_flow = val_gen.flow_from_dataframe(
    val_df, x_col='path', y_col='label',
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

NUM_CLASSES = len(train_flow.class_indices)
print(f"Classes ({NUM_CLASSES}) :", train_flow.class_indices)

In [ ]:
import numpy as np
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from sklearn.utils.class_weight import compute_class_weight

# Poids de classe
labels_array = train_df['label'].map(train_flow.class_indices).values
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=labels_array
)
class_weights = dict(enumerate(class_weights_arr))
print("Poids :", class_weights)

# Architecture
base_model = EfficientNetB4(weights='imagenet', include_top=False,
                             input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)  # ← softmax multi-classes

model = Model(inputs, outputs)
print(f"Paramètres entraînables : {model.count_params():,}")

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',   # ← multi-classes
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks_1 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5,
                                      restore_best_weights=True, mode='max'),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/MycoVision/best_head.keras',
        monitor='val_accuracy', save_best_only=True, mode='max')
]

print("Étape 1 — Entraînement de la tête...")
history1 = model.fit(
    train_flow, epochs=15, validation_data=val_flow,
    class_weight=class_weights, callbacks=callbacks_1
)

In [ ]:
base_model.trainable = True
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

callbacks_2 = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=7,
                                      restore_best_weights=True, mode='max'),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3),
    tf.keras.callbacks.ModelCheckpoint(
        '/content/drive/MyDrive/MycoVision/best_finetuned.keras',
        monitor='val_accuracy', save_best_only=True, mode='max')
]

print("Étape 2 — Fine-tuning complet...")
history2 = model.fit(
    train_flow, epochs=30, validation_data=val_flow,
    class_weight=class_weights, callbacks=callbacks_2
)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

val_flow.reset()
y_pred_proba = model.predict(val_flow, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = val_flow.classes

label_names = list(train_flow.class_indices.keys())
print(classification_report(y_true, y_pred, target_names=label_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=label_names, yticklabels=label_names)
plt.title('Matrice de confusion — Champignons')
plt.ylabel('Réel'); plt.xlabel('Prédit')
plt.xticks(rotation=45); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# Sauvegarde les labels dans l'ordre du modèle
labels_path = '/content/drive/MyDrive/MycoVision/labels.txt'
with open(labels_path, 'w') as f:
    for label in label_names:
        f.write(label + '\n')
print("Labels sauvegardés :", label_names)

# Export TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
converter._experimental_lower_tensor_list_ops = False
tflite_model = converter.convert()

tflite_path = '/content/drive/MyDrive/MycoVision/mycoVision.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f"Modèle exporté : {len(tflite_model)/1024/1024:.1f} MB")

# Vérification
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()
out = interpreter.get_output_details()
test = np.random.randint(0, 255, (1, 224, 224, 3)).astype(np.float32)
interpreter.set_tensor(inp[0]['index'], test)
interpreter.invoke()
scores = interpreter.get_tensor(out[0]['index'])[0]
print(f"Sortie shape : {scores.shape}  ← doit être ({NUM_CLASSES},)")
print(f"Scores : {scores}")
print(f"Classe prédite : {label_names[np.argmax(scores)]}")

In [ ]:
from google.colab import files
files.download(tflite_path)
files.download(labels_path)